In [ ]:
from feature_extraction import run_extraction, extract_all_features
from model import SVM, KNN, save_model, load_model
from config import (KERNEL, C, GAMMA, K, SEED, DURATIONS, FOLDER_PATH, TEST_SNR, TEST_COMPRESSION, DOWN_SR)
from evaluation import (grid_search, search_best_duration, evaluate_models, plot_robustness, show_table, show_confusion_grid)
from preprocessing import prepare_data, load_dataset, split_data, SPLITS, align_onset_all
import numpy as np
from tqdm import tqdm
import pandas as pd

from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

In [ ]:
original_dataset = dict(zip(SPLITS, split_data(load_dataset(FOLDER_PATH))))
original_dataset["train"].signals = align_onset_all(original_dataset["train"].signals)
original_features = extract_all_features(original_dataset["train"], desc="Extract Original Train Data")

In [ ]:
SVM_params, _ = grid_search(
    {"kernel": KERNEL, "C": C, "gamma": GAMMA},
    SVM,
    X= original_features.X, y= original_features.labels, 
    excludes=[(("kernel", "gamma"), ("linear", "scale"))],  # gamma has no effect on the linear kernel
    random_state=SEED
)

In [ ]:
KNN_params, _ = grid_search(
    {"k": K},
    KNN,
    X= original_features.X, y= original_features.labels
)

In [ ]:
svm_best_duration, svm_durations = search_best_duration(DURATIONS, SVM(random_state=SEED, **SVM_params), original_dataset["train"])

In [ ]:
knn_best_duration, knn_durations = search_best_duration(DURATIONS, KNN(**KNN_params), original_dataset["train"])

In [ ]:
print (svm_durations)
print (knn_durations)

accuracy_mean = {}
best_accuracy = 0
for key in svm_durations.keys():
    accuracy_mean[key] = (svm_durations[key] + knn_durations [key])/2

print(accuracy_mean)

best_duration, best_accuracy = max(accuracy_mean.items(), key=lambda item: item[1])

print(f"The best overall duration is {best_duration}s, with accuracy mean of {best_accuracy}")



In [ ]:
train, test = prepare_data(max_duration=best_duration)
data = {
    "train":train,
    "test": test,
}

# original, noise, mp3, downsample
original, noise, mp3, downsample = run_extraction(False, data)

testing = {
    "original": original["test"],
    "noise": noise["test"],
    "mp3": mp3["test"],
    "downsample": downsample["test"]
}

scenarios = {
    "original":              [original["train"]],
    "noise":                 [noise["train"]],
    "mp3":                   [mp3["train"]],
    "downsample":            [downsample["train"]],
    "original+noise":        [original["train"], noise["train"]],
    "original+mp3":          [original["train"], mp3["train"]],
    "original+noise+mp3":    [original["train"], noise["train"], mp3["train"]],
    "all":                   [original["train"], noise["train"], mp3["train"], downsample["train"]],
}

In [ ]:
svm_models = []
for scenario, data in tqdm(scenarios.items(), "Train SVM model for each scenario"):
    X_train = np.concatenate([part.X for part in data])
    y_train = np.concatenate([part.labels for part in data])

    svm_model = SVM(random_state=SEED, **SVM_params).fit(X_train, y_train)
    save_model(svm_model, (scenario+"_svm"))
    svm_models.append(svm_model)

In [ ]:
knn_models = []
for scenario, data in tqdm(scenarios.items(), "Train KNN model for each scenario"):
    X_train = np.concatenate([part.X for part in data])
    y_train = np.concatenate([part.labels for part in data])

    knn_model = KNN(**KNN_params).fit(X_train, y_train)
    save_model(knn_model, (scenario+"_knn"))
    knn_models.append(knn_model)

In [ ]:
svm_models = {scenario: load_model(scenario + "_svm") for scenario in scenarios}
knn_models = {scenario: load_model(scenario + "_knn") for scenario in scenarios}

# Evaluate every model on every test set and level
svm_results, svm_matrices = evaluate_models(svm_models, testing, "SVM")
knn_results, knn_matrices = evaluate_models(knn_models, testing, "KNN")
results = pd.concat([svm_results, knn_results], ignore_index=True)
matrices = {**svm_matrices, **knn_matrices}

# Store
results_dir = Path("pipeline_output/results")
results_dir.mkdir(parents=True, exist_ok=True)
results.to_csv(results_dir / "results.csv", index=False)
pd.to_pickle(matrices, results_dir / "confusion_matrices.pkl")

for model_name in ("SVM", "KNN"):
    display(Markdown(f"## {model_name}"))
    show_table(model_name, "macro_f1", results)
    show_table(model_name, "accuracy", results)

    plot_robustness(results, model_name, "macro_f1")
    plot_robustness(results, model_name, "accuracy")
    plt.show()
    
    show_confusion_grid(matrices, model_name, "original")       # clean test clips
    
    for snr in TEST_SNR:
        show_confusion_grid(matrices, model_name, "noise", snr)       # noise test clips
    for q in TEST_COMPRESSION:
        show_confusion_grid(matrices, model_name, "mp3", q) # mp3 test clips
    for sr in DOWN_SR:
        show_confusion_grid(matrices, model_name, "downsample", sr) # downsample test clips